## Import packages

In [ ]:
from pylab import *
from astropy.io import fits, ascii
from PyAstronomy import pyasl

## Read the FITS file

In [ ]:
###################################################
# for your test star, and to get started with WS1, use this:
# (comment/uncomment the relevant lines)

hdul = fits.open('../../../data/he1158-2313_uncor.fits')
head = hdul[1].header
data = hdul[1].data
# data # to check on the file structure
# # check what the two arrays are called, then pick option 1 or option 2: 

#Wavelength, Flux = data['wavelength'],data['flux']       #For some stars - option 1 
Wavelength, Flux = data['wavelength'],data['norm_flux']  #For other stars - option 2 


###################################################
# once you have your normalized SMH exported file, 
# use this to redo your line plots:  
# (comment/uncomment the relevant lines)

# tab = ascii.read('../../../data/he1158-2313_rest.txt')   # the WS3 route: our rest-frame ASCII
# Wavelength, Flux = tab['col1'],tab['col2']


## Plotting commands

In [ ]:
plt.plot(Wavelength, Flux)

## Customizing the plot

In [ ]:
fig  = plt.figure(figsize=(8,6)) 

plt.plot(Wavelength, Flux)

plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)


In [ ]:
fig  = plt.figure(figsize=(8,6)) 

plt.plot(Wavelength, Flux)
plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)


plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)



In [ ]:
fig  = plt.figure(figsize=(8,6)) 

plt.plot(Wavelength, Flux)
plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)
plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)


xlim([5100,5200])
ylim([0.0,1.1])


In [ ]:
fig  = plt.figure(figsize=(8,6)) 

#plt.plot(Wavelength, Flux-0.3, 'r.-' ,label="HD 122563") # data/HD122563.fits — you can choose different colors or even overplot with an offset
plt.plot(Wavelength, Flux, 'blue' ,label="HE 1158-2313")

plt.tick_params(axis="both",bottom=True, top=True, left=True, right=True, length=7,direction="in")
plt.tick_params(which='minor',direction="in",bottom=True, top=True, left=True, right=True, length=4, color='r')
plt.minorticks_on()
plt.tick_params(axis="both",which='major', direction="in", labelsize=15)
plt.ylabel('Flux', fontsize=14)
plt.xlabel('$\lambda$ ($\AA$)', fontsize=14)
xlim([5100,5200])
ylim([0.0,1.1])


plt.legend(loc="lower right", fontsize=16)
plt.savefig("plots/Pt3_Mglines.pdf")

## §3.2–3.3 — Comparing the spectrum with the Sun

In [ ]:
import numpy as np

D, P = "../../../data/", "plots/"
C = 299792.458
# The §4.2.2 velocity, read from Part 4's result rather than typed in here, so the two
# notebooks cannot drift apart.
import json
V_GEO = json.load(open("../part4-5_radial_velocity/results/"
                       "part4-2-2_geocentric_rv_HD122563.json"))["v_geo_kms"]  # adopted template
print(f"using v_geo = {V_GEO:+.2f} km/s (§4.2.2)")

# our star, as observed (AF's cells above already loaded these, but be explicit)
t_obs = fits.open(D + "he1158-2313_uncor.fits")[1].data
w_obs = np.asarray(t_obs["wavelength"], float)
f_obs = np.asarray(t_obs["norm_flux"], float)
o = np.argsort(w_obs, kind="stable")            # merged orders are not strictly monotonic
w_obs, f_obs = w_obs[o], f_obs[o]
w_rest = w_obs / (1 + V_GEO / C)                # the §4.3 shift

# the Sun, at rest, wavelengths in air
t_sun = fits.open(D + "solar_fts_kpno.fits")[1].data
w_sun = np.asarray(t_sun["wavelength"], float)
f_sun_native = np.asarray(t_sun["norm_flux"], float)

# The atlas is R ~ 400,000; MIKE is R ~ 28,000. Left native, the Sun looks artificially
# sharper and deeper than our star, which is a resolution artefact rather than physics.
# Smooth it to MIKE's resolution so depths and widths are comparable by eye.
from scipy.ndimage import gaussian_filter1d
R_MIKE = 28000.0
step = np.median(np.diff(w_sun))
sigma_px = (np.median(w_sun) / R_MIKE) / 2.355 / step
f_sun = gaussian_filter1d(f_sun_native, sigma_px)
print(f"solar atlas smoothed to R~{R_MIKE:.0f} (sigma {sigma_px:.1f} px, step {step:.4f} A)")

print(f"star: {len(w_obs)} px, {w_obs[0]:.0f}-{w_obs[-1]:.0f} A")
print(f"Sun : {len(w_sun)} px, {w_sun[0]:.0f}-{w_sun[-1]:.0f} A (NSO/Kitt Peak FTS)")
print(f"expected shift at 5183 A: {5183.62 * V_GEO / C:+.2f} A")

# G64-12, the §4.2.4 RV standard, as a second comparison. Both it and our star are MIKE spectra
# at comparable resolution (R ~ 22000 and ~17000), so unlike the solar atlas nothing is smoothed.
from PyAstronomy import pyasl
w_g64, f_g64 = (np.asarray(x, float) for x in pyasl.read1dFitsSpec(D + "G64-12.fits"))
print(f"G64-12 {len(w_g64)} px, {w_g64[0]:.0f}-{w_g64[-1]:.0f} A")

# the two reference stars AF names for §4.2.2, also MIKE spectra already at rest
w_hd140, f_hd140 = (np.asarray(x, float) for x in pyasl.read1dFitsSpec(D + "HD140283.fits"))
w_hd122, f_hd122 = (np.asarray(x, float) for x in pyasl.read1dFitsSpec(D + "HD122563.fits"))
print(f"HD 140283 {len(w_hd140)} px | HD 122563 {len(w_hd122)} px")

### The comparison spectra, side by side

In [ ]:
# Wide-context side-by-side: the comparison star on top of ours, one pair of rows per window.
# Narrow windows, not 500-A sweeps: with a 500 A row the panel is so wide and short that
# individual lines vanish. ~120 A each keeps the rows tall enough to read.
BANDS = [(3925, 4045), (4290, 4400), (5140, 5260), (5850, 5960)]
LABEL = {3933.66: "Ca II K", 3968.47: "Ca II H", 4101.73: r"H$\delta$", 4226.73: "Ca I",
         4340.46: r"H$\gamma$", 4861.35: r"H$\beta$", 5172.70: "Mg b", 5889.95: "Na D",
         6562.80: r"H$\alpha$"}


def overview(w_t, f_t, tname, tcol, out):
    """Paired rows: template above, our star below, sharing the wavelength axis.

    hspace=0 *inside* each pair puts the vertical space into the panels instead of the gaps.
    """
    short = tname.split(" (")[0]
    fig = plt.figure(figsize=(10.5, 11))
    outer = fig.add_gridspec(len(BANDS), 1, hspace=0.30, top=0.935, bottom=0.05,
                             left=0.075, right=0.985)
    for b, (lo, hi) in enumerate(BANDS):
        inner = outer[b].subgridspec(2, 1, hspace=0.0)
        rows = [(w_t, f_t, tname, tcol), (w_obs, f_obs, "HE 1158-2313 (as observed)", "C0")]
        for k, (wx, fx, name, col) in enumerate(rows):
            ax = fig.add_subplot(inner[k])
            m = (wx > lo) & (wx < hi)
            xb, yb = wx[m].copy(), fx[m].copy()
            yb[np.r_[np.diff(xb) > 0.5, False]] = np.nan      # never draw across a gap
            ax.plot(xb, yb, color=col, lw=0.5)
            for lam, txt in LABEL.items():
                if lo < lam < hi:
                    lam_here = lam if k == 0 else lam * (1 + V_GEO / C)
                    ax.axvline(lam_here, color="0.7", lw=0.6, ls="--", zorder=0)
                    if k == 0:
                        ax.text(lam_here, 1.04, txt, ha="center", va="bottom",
                                fontsize=8, color="0.35")
            ax.set_xlim(lo, hi)
            ax.set_ylim(0, 1.32)
            ax.set_yticks([0.0, 0.5, 1.0])
            ax.set_ylabel(short if k == 0 else "HE", fontsize=8.5)
            ax.minorticks_on()
            ax.tick_params(direction="in", top=True, right=True, which="both", labelsize=8)
            if k == 0:
                ax.set_xticklabels([])
            elif b == len(BANDS) - 1:
                ax.set_xlabel(r"$\lambda$ ($\AA$)", fontsize=9)
            ax.text(0.995, 0.06, name, transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=7.5, color=col,
                    bbox=dict(boxstyle="square,pad=0.15", fc="white", ec="none", alpha=0.85))
    fig.suptitle(f"{short} vs HE 1158-2313 — paired rows, same normalisation."
                 "\nMarked: Ca II H & K, the G band, H$\\gamma$, Mg b, Na D. Grey dashes are rest "
                 "wavelengths above, the redshifted position below.",
                 fontsize=10, x=0.012, ha="left")
    fig.savefig(out, dpi=130)
    display(fig)
    plt.close(fig)
    print("wrote", out)


# the solar atlas is smoothed to MIKE's resolution (see the loader); G64-12 needs no smoothing
overview(w_sun,   f_sun,   "Sun (NSO/KP FTS, at rest)", "0.35", P + "overview_sun_vs_star.png")
overview(w_hd122, f_hd122, "HD 122563 (at rest)",       "C4",   P + "overview_hd122563_vs_star.png")
overview(w_hd140, f_hd140, "HD 140283 (at rest)",       "C1",   P + "overview_hd140283_vs_star.png")
overview(w_g64,   f_g64,   "G64-12 (at rest)",          "C2",   P + "overview_g64-12_vs_star.png")

## §3.2–3.4 — Rest-frame line regions

In [ ]:
def cut(lo, hi, wave=None, flux=None):
    wave = w_obs if wave is None else wave
    flux = f_obs if flux is None else flux
    m = (wave > lo) & (wave < hi)
    return wave[m], flux[m]

def panel(ax, lo, hi, labels, title, both=False):
    x, y = cut(lo, hi, w_rest, f_obs); ax.plot(x, y, "k", lw=0.8, label="rest frame" if both else None)
    if both:
        x0, y0 = cut(lo, hi); ax.plot(x0, y0, "C0", lw=0.8, alpha=0.7, label="observed (uncorrected)")
    for lam, lab in labels.items():
        ax.axvline(lam, color="C3", lw=0.6, ls="--"); ax.text(lam, 1.06, lab, rotation=90, ha="center", va="bottom", fontsize=8, color="C3")
    ax.set_xlim(lo, hi); ax.set_ylim(0, 1.25); ax.set_xlabel("Wavelength [Å]"); ax.set_ylabel("Normalised flux"); ax.set_title(title, fontsize=10, loc="left")
    ax.minorticks_on(); ax.tick_params(direction="in", top=True, right=True, which="both")
    if both: ax.legend(loc="lower left", fontsize=8, frameon=False)

mg = {5167.321: "Mg I", 5172.698: "Mg I", 5183.619: "Mg I"}
fig, ax = plt.subplots(figsize=(9, 3.6)); panel(ax, 5130, 5190, mg, "Mg b triplet — observed vs. rest frame", both=True); fig.tight_layout(); fig.savefig(P + "mgb_observed_vs_rest.png"); display(fig); plt.close(fig)
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
panel(axes[0, 0], 5130, 5190, mg, "(a) Mg b"); panel(axes[0, 1], 4310, 4325, {4312.86: "Ti II", 4314.97: "Ti II", 4318.65: "Ca I", 4320.73: "Sc II"}, "(b) CH G band (4300-4325: many blended CH A-X lines)")
panel(axes[1, 0], 4074, 4089, {4077.709: "Sr II", 4084.49: "Fe I"}, "(c) Sr II 4077"); panel(axes[1, 1], 4126, 4133, {4129.72: "Eu II", 4127.61: "Fe I", 4132.06: "Fe I"}, "(d) Eu II 4129")
fig.tight_layout(); fig.savefig(P + "multipanel_lines_rest.png"); display(fig); plt.close(fig)
fig, ax = plt.subplots(figsize=(11, 3.2)); ax.plot(w_rest, f_obs, "k", lw=0.3); ax.set_ylim(0, 1.3); ax.set_xlim(3600, 9000)
ax.set_xlabel("Wavelength [Å]"); ax.set_ylabel("Normalised flux"); ax.set_title("HE 1158-2313 — full MIKE spectrum, rest frame", loc="left", fontsize=10); fig.tight_layout(); fig.savefig(P + "full_spectrum_rest.png"); display(fig); plt.close(fig)
print("  figures -> " + P)